In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.13.0+cu130
13.0
True
NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
import json
import random
import sys
from pathlib import Path
import numpy as np

current_dir = Path.cwd().resolve()
WORKSPACE_ROOT = next(
    (candidate for candidate in (current_dir, *current_dir.parents)
     if (candidate / "Temporal-Testing").is_dir() and (candidate / "Fin-RATE").is_dir()),
    None,
)
if WORKSPACE_ROOT is None:
    raise FileNotFoundError("Could not locate the Inception workspace from the notebook directory.")
TEMPORAL_TESTING_DIR = WORKSPACE_ROOT / "Temporal-Testing"
if str(TEMPORAL_TESTING_DIR) not in sys.path:
    sys.path.insert(0, str(TEMPORAL_TESTING_DIR))

from reproducibility import DEFAULT_SEED, seed_everything
SEED = DEFAULT_SEED
seed_everything(SEED)

42

In [3]:
with (WORKSPACE_ROOT / "Fin-RATE" / "qa" / "LT-QA.json").open("r", encoding="utf-8") as file:
    ltqa = json.load(file)

In [4]:
nqa = 100
smoke_test_ltqa = random.sample(ltqa,nqa)
sample_docs = set()
for sample_qa in smoke_test_ltqa:
    docs = set(sample_qa["doc_ids"])
    sample_docs.update(docs)

#print(sample_docs)
print(len(sample_docs))

218


In [5]:
len(ltqa)

2500

In [6]:
#System 1: Normal Vector Retrieval
from vector_retrieval import retrieval_pipeline
recalls = []
mrr = 0
for qa in smoke_test_ltqa:
    chunks = retrieval_pipeline(qa["question"], seed=SEED)
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set(list(set(chunks))[:5]) & set(golden))/len(golden)
    recall10 = len(set(list(set(chunks))[:10]) & set(golden))/len(golden)

    recalls.append((recall5,recall10))

    rr = 0
    for rank, item in enumerate(chunks, start=1):
        if item in golden:
            rr = 1 / rank
            break
    mrr += rr

#Combine chunks of same docs and then calc recall

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))
#print(mrr/nqa)


0.0675
0.14116666666666666


In [7]:
#System 2: ChromaDB Vector Retrieval
from ChromaSetup import create_embedding_function, build_chroma_database, retrieve_relevant_chunks



In [1]:
!{sys.executable} "{TEMPORAL_TESTING_DIR / 'ChromaSetup.py'}" --build --embedding-backend bge --device cuda --seed {SEED}

^C


In [10]:
recalls = []
for qa in smoke_test_ltqa:
    chunks = retrieve_relevant_chunks(qa["question"], seed=SEED)
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set(list(set([chun['metadata']['doc_id'] for chun in chunks]))[:5]) & set(golden))/len(golden)
    recall10 = len(set(list(set([chun['metadata']['doc_id'] for chun in chunks]))[:10]) & set(golden))/len(golden)
    recalls.append((recall5,recall10))

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))

0.0
0.0


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-m3", device=device)

embeddings = model.encode(
    ["What is reranking in RAG?"],
    batch_size=8,
    normalize_embeddings=True,
)
print(embeddings.shape)

In [16]:
#System 3: CRAG 
'''

from ChromaSetup import create_embedding_function, build_chroma_database, retrieve_relevant_chunks

ef = create_embedding_function(
    backend="sentence-transformers",
    device="cuda",
    seed=SEED,
)

recalls = []
for qa in smoke_test_ltqa:
    chunks = retrieve_relevant_chunks(qa["question"], seed=SEED)
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set([chun['metadata']['doc_id'] for chun in chunks][:5]) & set(golden))/len(golden)
    recall10 = len(set([chun['metadata']['doc_id'] for chun in chunks][:10]) & set(golden))/len(golden)
    recalls.append((recall5,recall10))

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))

'''

'\n\nfrom ChromaSetup import create_embedding_function, build_chroma_database, retrieve_relevant_chunks\n\nef = create_embedding_function(\n    backend="sentence-transformers",\n    device="cuda",\n)\n\nrecalls = []\nfor qa in smoke_test_ltqa:\n    chunks = retrieve_relevant_chunks(qa["question"])\n    golden = qa["doc_ids"]\n    #Recall 5\n    recall5 = len(set([chun[\'metadata\'][\'doc_id\'] for chun in chunks][:5]) & set(golden))/len(golden)\n    recall10 = len(set([chun[\'metadata\'][\'doc_id\'] for chun in chunks][:10]) & set(golden))/len(golden)\n    recalls.append((recall5,recall10))\n\nprint(np.mean([recs[0] for recs in recalls]))\nprint(np.mean([recs[1] for recs in recalls]))\n\n'